In [1]:
import os
from pathlib import Path


In [2]:
%pwd

'c:\\Users\\asdaw\\Desktop\\medical-dcgan\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\asdaw\\Desktop\\medical-dcgan'

In [5]:
import sys
print(sys.executable)


c:\Users\asdaw\Desktop\medical-dcgan\dcgan_proj\Scripts\python.exe


In [6]:
%pwd

'c:\\Users\\asdaw\\Desktop\\medical-dcgan'

In [7]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_dir: Path
    local_data_file: Path
    unzip_dir: Path


In [8]:
from dcGAN_image_generation.constants import *
from dcGAN_image_generation.utils.common import read_yaml, create_directories


In [9]:

class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        self.artifacts_root = ROOT_DIR / self.config.artifacts_root
        create_directories([self.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:

        config = self.config.data_ingestion

        root_dir = ROOT_DIR / config.root_dir

        create_directories([root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=root_dir,
            source_dir=ROOT_DIR / config.source_dir,
            local_data_file=ROOT_DIR / config.local_data_file,
            unzip_dir=ROOT_DIR / config.unzip_dir
        )

        return data_ingestion_config

In [10]:
import os
import shutil
from dcGAN_image_generation import logger
from dcGAN_image_generation.utils.common import get_size
from pathlib import Path


In [11]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def copy_data(self) -> None:
        """
        Copies dataset from source_dir → artifacts ingestion folder
        """

        try:
            source = Path(self.config.source_dir)
            destination = Path(self.config.unzip_dir)

            logger.info(f"Source directory: {source}")
            logger.info(f"Destination directory: {destination}")

            if not source.exists():
                raise FileNotFoundError(f"Source data not found at {source}")

            if destination.exists():
                logger.info("Data already ingested. Skipping copy.")
                return

            shutil.copytree(src=source, dst=destination)

            logger.info(
                f"Data successfully ingested from {source} to {destination}"
            )

            logger.info(f"Ingested data size: {get_size(destination)}")

        except Exception as e:
            raise e


In [12]:
print("Config path:", CONFIG_FILE_PATH)
print("Exists:", Path(CONFIG_FILE_PATH).exists())

Config path: config\config.yaml
Exists: True


In [13]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.copy_data()
except Exception as e:
    raise e

[2026-02-19 07:49:52,928: INFO: common]: yaml file loaded: config\config.yaml]
[2026-02-19 07:49:52,930: INFO: common]: yaml file loaded: params.yaml]
[2026-02-19 07:49:52,932: INFO: common]: created directory at: C:\Users\asdaw\Desktop\medical-dcgan\artifacts]
[2026-02-19 07:49:52,934: INFO: common]: created directory at: C:\Users\asdaw\Desktop\medical-dcgan\artifacts\data_ingestion]
[2026-02-19 07:49:52,935: INFO: 3552728949]: Source directory: C:\Users\asdaw\Desktop\medical-dcgan\data\chest_xray]
[2026-02-19 07:49:52,935: INFO: 3552728949]: Destination directory: C:\Users\asdaw\Desktop\medical-dcgan\artifacts\data_ingestion\chest_xray]
[2026-02-19 07:50:47,020: INFO: 3552728949]: Data successfully ingested from C:\Users\asdaw\Desktop\medical-dcgan\data\chest_xray to C:\Users\asdaw\Desktop\medical-dcgan\artifacts\data_ingestion\chest_xray]
[2026-02-19 07:50:47,022: INFO: 3552728949]: Ingested data size: ~ 0 KB]
